# Problema X23B11T0Y21B21 — Problema 2D pelo Produto de Funções de Green

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ana-mat-br/codigos-livro-conducao-calor/blob/main/x23b11t0y21b21.ipynb)

**Livro:** *Condução de Calor: Aplicações das Funções de Green em Problemas de Engenharia*
**Autores:** Ana Paula Fernandes e Gilmar Guimarães

---

## Descrição do problema

Placa retangular $0 \le x \le a$, $0 \le y \le b$, com condições de contorno independentes em cada direção:

- **$x = 0$** (tipo 2): fluxo de calor prescrito $q''_0$
- **$x = a$** (tipo 3): convecção com coeficiente $h$ e meio a $T_\infty$
- **$y = 0$** (tipo 2): fluxo de calor $q''_1(t) = q''_1 + A_0\,t$ (linear no tempo)
- **$y = b$** (tipo 1): temperatura prescrita $C$
- **Condição inicial:** $T(x, y, 0) = 0$

A notação **X23…Y21…** resume as condições: em $x$, fluxo em $x=0$ (2) e convecção em $x=a$ (3); em $y$, fluxo em $y=0$ (2) e temperatura em $y=b$ (1).

## Solução analítica

Pelo **produto de funções de Green transientes**, a temperatura é a superposição de cinco parcelas, uma para cada fonte não-homogênea:

$$T(x,y,t) = C + \Theta_{00} + \Theta_{1} + \Theta_{2} + \Theta_{30} + \Theta_{31}$$

- $\Theta_{00}$ — resposta à condição inicial homogeneizada;
- $\Theta_{1}$ — fluxo imposto em $x=0$;
- $\Theta_{2}$ — convecção em $x=a$;
- $\Theta_{30}$ — parcela constante do fluxo em $y=0$;
- $\Theta_{31}$ — parcela linear no tempo ($A_0 t$) do fluxo em $y=0$.

Cada parcela é uma série dupla nos autovalores $\beta_m$ (direção $x$) e $\gamma_n$ (direção $y$), com o autovalor combinado

$$\lambda_{mn} = \left(\frac{\beta_m}{a}\right)^2 + \left(\frac{\gamma_n}{b}\right)^2 .$$

### Autovalores

- **Direção $x$** (convecção): os $\beta_m$ são raízes da equação transcendental $\beta\tan\beta = B$, com $B = ha/k$ — obtidas numericamente pelo **método de Brent**.
- **Direção $y$** (fluxo + temperatura): têm **forma fechada**, $\gamma_n = \left(n - \tfrac{1}{2}\right)\pi$.

## Bibliotecas utilizadas

- **NumPy** (`numpy`): vetores, matrizes e funções matemáticas; as séries duplas são somadas de forma vetorizada com `np.einsum`.
- **Matplotlib** (`matplotlib.pyplot`): gráficos.
- **SciPy** (`scipy.optimize.brentq`): método de Brent para as raízes dos autovalores $\beta_m$ na direção $x$.

In [ ]:
import numpy as np                        # computação numérica
import matplotlib.pyplot as plt           # gráficos
from scipy.optimize import brentq         # raízes de funções (método de Brent)

## Parâmetros do problema

Os parâmetros geométricos e físicos são definidos abaixo. O número de termos das séries, $M$ (em $x$) e $N$ (em $y$), controla o truncamento — 80 termos em cada direção bastam para este problema. O parâmetro adimensional $B = ha/k$ mede o peso da convecção em $x=a$.

In [ ]:
a     = 66e-4        # dimensão da placa em x [m]
b     = 66e-4        # dimensão da placa em y [m]
alpha = 1.93e-7      # difusividade térmica [m²/s]
k     = 0.81         # condutividade térmica [W/(m·K)]
q0    = 1e4          # fluxo imposto em x=0 [W/m²]
q1    = 0.0          # parcela constante do fluxo em y=0 [W/m²]
A0    = 5.0          # coeficiente do fluxo linear no tempo em y=0 [W/(m²·s)]
h     = 5.0          # coeficiente convectivo em x=a [W/(m²·K)]
Tinf  = 200.0        # temperatura do meio convectivo [°C]
C     = 30.0         # temperatura prescrita em y=b [°C]

M = 80               # número de termos da série na direção x
N = 80               # número de termos da série na direção y

B2 = h * a / k       # parâmetro adimensional de Biot na direção x
print(f'B = h·a/k = {B2:.4f}')

## Autovalores das duas direções

Na direção $y$ os autovalores têm forma fechada, $\gamma_n = (n-\tfrac12)\pi$. Na direção $x$, a condição convectiva leva à equação transcendental $\beta\tan\beta = B$, sem solução explícita: cada raiz $\beta_m$ é buscada no seu ramo da tangente, no intervalo $\big((m-1)\pi,\ (m-1)\pi + \pi/2\big)$, pela função `brentq`.

Quando $B$ é muito pequeno (convecção quase nula — usada nas verificações adiante), a primeira raiz fica tão próxima de zero que escapa da resolução numérica do extremo do intervalo; nesse caso usa-se a expansão assintótica $\beta_1 \approx \sqrt{B}$ e $\beta_m \approx (m-1)\pi + B/[(m-1)\pi]$.

In [ ]:
def autovalores_x23(B2, M):
    """M primeiras raízes de β·tan(β) = B2, uma por ramo ((m-1)π, (m-1)π + π/2)."""
    raizes = np.empty(M)
    for m in range(1, M + 1):
        lo = (m - 1) * np.pi
        try:
            raizes[m - 1] = brentq(lambda bt: bt * np.tan(bt) - B2,
                                   lo + 1e-12, lo + np.pi / 2 - 1e-12)
        except ValueError:
            # B2 muito pequeno: raiz além da resolução do extremo -> assintótica
            raizes[m - 1] = np.sqrt(B2) if m == 1 else lo + B2 / lo
    return raizes

beta = autovalores_x23(B2, M)                     # autovalores em x (numéricos)
gama = np.pi * (np.arange(1, N + 1) - 0.5)        # autovalores em y: (n-1/2)π (forma fechada)

print('Primeiros 5 autovalores β_m (direção x):', np.round(beta[:5], 5))
print('Primeiros 5 autovalores γ_n (direção y):', np.round(gama[:5], 5))

## Cálculo da temperatura

A função abaixo monta as cinco parcelas. Cada uma é uma série dupla somada com `np.einsum('mn,mnt->t', coef, E)`, que multiplica os coeficientes $(m,n)$ pelo fator temporal $E_{mnt}$ e soma sobre $m$ e $n$, devolvendo um vetor no tempo. O fator temporal é $E = e^{-\lambda_{mn}\alpha t}$ para a condição inicial e $1 - E$ para as fontes de contorno constantes.

Ao final, verificamos numericamente a condição inicial ($T \approx 0$ em $t \approx 0$) e o contorno de temperatura prescrita ($T = C$ em $y = b$).

In [ ]:
def solucao_x23y21(x, y, t, *, a, b, alpha, k, q0, q1, A0, h, Tinf, C, M, N):
    """Temperatura T(x, y, t) do Problema X23B11T0Y21B21 (x, y escalares; t vetor)."""
    B2   = h * a / k
    beta = autovalores_x23(B2, M)                  # autovalores em x
    gama = np.pi * (np.arange(1, N + 1) - 0.5)     # autovalores em y (forma fechada)

    Nbm  = (beta**2 + B2**2) / (beta**2 + B2**2 + B2)   # normalização em x

    Bm   = beta[:, None]                           # (M,1)
    Gn   = gama[None, :]                            # (1,N)
    Dmn  = (Bm / a)**2 + (Gn / b)**2               # autovalor combinado λ_mn (M,N)

    cosx = np.cos(Bm * x / a)
    cosy = np.cos(Gn * y / b)
    sinb = np.sin(Bm)
    sing = np.sin(Gn)

    Theta0   = -C                                  # condição inicial homogeneizada
    ThetaInf = Tinf - C

    base = Nbm[:, None] * cosx * cosy              # (M,N)
    Et   = np.exp(-Dmn[..., None] * alpha * t[None, None, :])   # e^{-λ_mn α t}  (M,N,T)

    # Θ00 — resposta à condição inicial
    coef00 = 4 * Theta0 * base * (sinb / Bm) * (sing / Gn)
    T00 = np.einsum('mn,mnt->t', coef00, Et)

    # Θ1 — fluxo imposto em x=0
    coef1 = (q0 / k) * (4 / a) * base * (sing / Gn) / Dmn
    T1 = np.einsum('mn,mnt->t', coef1, 1 - Et)

    # Θ2 — convecção em x=a
    coef2 = (h * ThetaInf / k) * (4 / a) * base * (sing / Gn) * np.cos(Bm) / Dmn
    T2 = np.einsum('mn,mnt->t', coef2, 1 - Et)

    # Θ30 — parcela constante do fluxo em y=0
    coef30 = (q1 / k) * (4 / b) * base * (sinb / Bm) / Dmn
    T30 = np.einsum('mn,mnt->t', coef30, 1 - Et)

    # Θ31 — parcela linear no tempo do fluxo em y=0
    coef31 = (A0 / k) * (4 / b) * base * (sinb / Bm)
    nucleo31 = (t[None, None, :] / Dmn[..., None]
                - 1 / (Dmn[..., None]**2 * alpha)
                + Et / (Dmn[..., None]**2 * alpha))
    T31 = np.einsum('mn,mnt->t', coef31, nucleo31)

    return C + T00 + T1 + T2 + T30 + T31

# Verificação da condição inicial e do contorno y=b
par = dict(a=a, b=b, alpha=alpha, k=k, q0=q0, q1=q1, A0=A0, h=h, Tinf=Tinf, C=C, M=M, N=N)
print(f'T(a/2, b/2, t≈0) = {solucao_x23y21(a/2, b/2, np.array([1e-4]), **par)[0]:8.4f} °C  (esperado ≈ 0)')
print(f'T(a/2, b,   t=50) = {solucao_x23y21(a/2, b, np.array([50.0]), **par)[0]:8.4f} °C  (esperado = C = {C})')

## Campo de temperatura transiente

O gráfico mostra a evolução de $T$ em cinco pontos: os três contornos em $x$ (na altura média $y=b/2$) e os dois contornos em $y$ (na largura média $x=a/2$). A paleta **Okabe-Ito**, segura para daltonismo, é o padrão adotado no livro.

In [ ]:
cores    = ['#D55E00', '#0072B2', '#009E73', '#CC79A7', '#E69F00']  # paleta Okabe-Ito
tracados = ['-', '--', '-.', ':', (0, (5, 1))]

t = np.linspace(0.01, 100.0, 500)

pontos = [
    (0.0, b/2, r'$x=0,\ y=b/2$'),
    (a/2, b/2, r'$x=a/2,\ y=b/2$'),
    (a,   b/2, r'$x=a,\ y=b/2$'),
    (a/2, 0.0, r'$x=a/2,\ y=0$'),
    (a/2, b,   r'$x=a/2,\ y=b$'),
]

fig, ax = plt.subplots()
for (x, y, rot), cor, ls in zip(pontos, cores, tracados):
    T = solucao_x23y21(x, y, t, **par)
    ax.plot(t, T, label=rot, color=cor, linestyle=ls, linewidth=1.5)
ax.set_xlabel('Tempo (s)')
ax.set_ylabel('T(x, y, t)  (°C)')
ax.set_title('Problema X23B11T0Y21B21 — campo de temperatura transiente')
ax.set_xlim(0, t[-1])
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## Verificação intrínseca — direção $x$

Uma solução multidimensional deve reproduzir soluções 1D conhecidas quando reduzida a elas. Anulando a convecção ($h \to 0$), alongando muito o domínio em $y$ ($b = 10a$) e removendo o fluxo em $y=0$, o ponto $x=0,\ y=b/2$ deve coincidir com o **Problema X22B10T0** (placa isolada com fluxo imposto, Cap. 3), cuja solução 1D é conhecida.

In [ ]:
b_ver = 10 * a          # domínio muito alongado em y
h_ver = 1e-7            # convecção praticamente nula em x=a
t_mk  = t[::14]         # instantes dos marcadores

par_x = dict(a=a, b=b_ver, alpha=alpha, k=k, q0=q0, q1=0.0, A0=0.0,
             h=h_ver, Tinf=0.0, C=0.0, M=400, N=400)
T2d = solucao_x23y21(0.0, b_ver/2, t_mk, **par_x)

# X22B10T0 (Cap. 3): forma fechada estacionária + série transiente, em x=0
mm    = np.arange(1, 201)
lam   = (mm * np.pi / a)**2 * alpha
serie = np.einsum('m,mt->t', 1/mm**2, np.exp(-np.outer(lam, t)))
Tx22  = (q0 * a / k) * (alpha * t / a**2 + 1/3 - (2/np.pi**2) * serie)

fig, ax = plt.subplots()
ax.plot(t, Tx22, color='#000000', linewidth=1.5, label=r'X22B10T0 (1D), $x=0$')
ax.plot(t_mk, T2d, color=cores[0], linestyle='none', marker='o', markersize=4.5,
        markerfacecolor='none', label=r'X23B11T0Y21B21 (2D), $x=0,\ y=b/2$')
ax.set_xlabel('Tempo (s)')
ax.set_ylabel('T  (°C)')
ax.set_title(r'Verificação na direção $x$:  $h \to 0$,  $b = 10a$')
ax.set_xlim(0, t[-1])
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

serie_mk = np.einsum('m,mt->t', 1/mm**2, np.exp(-np.outer(lam, t_mk)))
Tx22_mk  = (q0 * a / k) * (alpha * t_mk / a**2 + 1/3 - (2/np.pi**2) * serie_mk)
print(f'Desvio máximo 2D vs X22B10T0: {np.max(np.abs(T2d - Tx22_mk)):.2e} °C')

## Verificação intrínseca — direção $y$

De forma análoga, anulando o fluxo em $x=0$, mantendo a convecção desprezível e impondo um fluxo constante em $y=0$ sobre um domínio muito profundo ($10b$), nos primeiros instantes o ponto $y=0$ não "sente" a fronteira oposta e se comporta como um **meio semi-infinito** — o **Problema X20B1T0** (Cap. 3), cuja temperatura de superfície é $T(0,t) = (q''_0/k)\sqrt{4\alpha t/\pi}$.

In [ ]:
q1_ver = 1e4            # fluxo constante em y=0 para a verificação
par_y = dict(a=a, b=10*b, alpha=alpha, k=k, q0=0.0, q1=q1_ver, A0=0.0,
             h=1e-5, Tinf=0.0, C=0.0, M=100, N=800)
T2d_y = solucao_x23y21(a/2, 0.0, t_mk, **par_y)

# X20B1T0 (Cap. 3): meio semi-infinito com fluxo imposto, na superfície y=0
Tx20 = (q1_ver / k) * np.sqrt(4 * alpha * t / np.pi)

fig, ax = plt.subplots()
ax.plot(t, Tx20, color='#000000', linewidth=1.5, label=r'X20B1T0 (semi-infinito), $y=0$')
ax.plot(t_mk, T2d_y, color=cores[1], linestyle='none', marker='s', markersize=4.5,
        markerfacecolor='none', label=r'2D, $x=a/2,\ y=0$')
ax.set_xlabel('Tempo (s)')
ax.set_ylabel('T  (°C)')
ax.set_title(r'Verificação na direção $y$:  $q_0=0$,  domínio $10b$')
ax.set_xlim(0, t[-1])
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

Tx20_mk = (q1_ver / k) * np.sqrt(4 * alpha * t_mk / np.pi)
print(f'Desvio máximo 2D vs X20B1T0: {np.max(np.abs(T2d_y - Tx20_mk)):.2e} °C')